In [32]:
%run bootstrap.ipynb

Exception: File `'bootstrap.ipynb.py'` not found.

In [ ]:
import os
import sys

# Clone thư viện aima-python nếu chưa có
if not os.path.exists('aima-python'):
    !git clone https://github.com/aimacode/aima-python.git

# Thêm thư mục vào sys.path để Python nhận diện module 'aima'
sys.path.append(os.path.abspath('aima-python'))

# Khởi tạo các thành phần tương tự như bootstrap.ipynb
from aima.games import *
from aima.notebook_utils import *

# TRÒ CHƠI HOẶC TÌM KIẾM ĐỐI KHÁNG

Notebook này đóng vai trò là tài liệu hỗ trợ cho các chủ đề được đề cập trong **Chương 5 - Tìm kiếm Đối kháng** trong cuốn sách *Artificial Intelligence: A Modern Approach.* Notebook này sử dụng các triển khai từ mô-đun [games.py](https://github.com/aimacode/aima-python/blob/master/games.py). Hãy import các class, phương thức, biến toàn cục cần thiết từ mô-đun games.

# NỘI DUNG

* Biểu diễn Trò chơi
* Các Ví dụ về Trò chơi
    * Tic-Tac-Toe
    * Trò chơi trong Hình 5.2
* Thuật toán Min-Max
* Cắt tỉa Alpha-Beta
* Người chơi (Players)
* Hãy cùng Chơi thử một số Trò chơi!

In [33]:
from aima.games import *
from aima.notebook_utils import psource, pseudocode

# BIỂU DIỄN TRÒ CHƠI

Để biểu diễn các trò chơi, chúng ta sử dụng lớp `Game`. Chúng ta có thể tạo lớp con kế thừa từ lớp này và ghi đè (override) các hàm của nó để biểu diễn trò chơi của riêng mình. Một công cụ hỗ trợ hữu ích là `GameState` (dưới dạng namedtuple), trong một số trường hợp sẽ rất tiện lợi, đặc biệt là khi trò chơi của chúng ta cần ghi nhớ một bàn cờ (như cờ vua).

## Namedtuple `GameState`

`GameState` là một [namedtuple](https://docs.python.org/3.5/library/collections.html#collections.namedtuple) biểu diễn trạng thái hiện tại của trò chơi. Nó được sử dụng để hỗ trợ biểu diễn các trò chơi mà trạng thái của chúng khó có thể biểu diễn theo cách thông thường, hoặc cho các trò chơi yêu cầu ghi nhớ bàn cờ, chẳng hạn như Tic-Tac-Toe.

`GameState` được định nghĩa như sau:

`GameState = namedtuple('GameState', 'to_move, utility, board, moves')`

* `to_move`: Đại diện cho lượt đi tiếp theo là của người chơi nào.

* `utility`: Lưu trữ giá trị hữu dụng (utility) của trạng thái trò chơi. Việc lưu trữ giá trị hữu dụng này là một ý tưởng hay, bởi vì khi thực hiện Tìm kiếm Minimax hoặc Tìm kiếm Alphabeta, bạn sẽ tạo ra nhiều lượt gọi đệ quy đi xuống tận cùng các trạng thái lá (trạng thái kết thúc). Khi các lượt gọi đệ quy này quay ngược trở lại hàm gọi ban đầu, chúng ta đã tính toán xong giá trị hữu dụng cho rất nhiều trạng thái. Chúng ta lưu trữ các giá trị này trong các đối tượng `GameState` tương ứng để tránh phải tính toán lại từ đầu.

* `board`: Một dictionary lưu trữ bàn cờ của trò chơi.

* `moves`: Lưu trữ danh sách các nước đi hợp lệ có thể thực hiện từ vị trí hiện tại.

## Lớp `Game`

Hãy cùng xem qua lớp `Game` trong mô-đun của chúng ta. Chúng ta thấy rằng nó có các hàm: `actions`, `result`, `utility`, `terminal_test`, `to_move` và `display`.

Lưu ý rằng các hàm này chưa thực sự được triển khai chi tiết. Lớp này chỉ đóng vai trò là một lớp mẫu (template class); chúng ta cần tạo lớp riêng cho trò chơi của mình bằng cách kế thừa lớp `Game` này và tự triển khai tất cả các phương thức được đề cập.

In [34]:
%psource Game

Bây giờ, hãy đi vào chi tiết của tất cả các phương thức trong lớp `Game` của chúng ta. Bạn phải triển khai các phương thức này khi tạo các lớp mới đại diện cho trò chơi của mình.

* `actions(self, state)`: Trả về danh sách hoặc generator chứa tất cả các nước đi hợp lệ từ trạng thái hiện tại. Việc trả về một generator thay vì một danh sách có ưu điểm là tiết kiệm bộ nhớ và bạn vẫn có thể thao tác với nó như một danh sách bình thường.

* `result(self, state, move)`: Nhận vào một trạng thái và một nước đi, trả về trạng thái mới nhận được sau khi thực hiện nước đi đó.

* `utility(self, state, player)`: Nhận vào một trạng thái kết thúc (terminal state) và một người chơi, trả về giá trị hữu dụng cho người chơi đó tại trạng thái kết thúc này. Khi triển khai phương thức này, hãy giả định rằng trạng thái truyền vào chắc chắn là trạng thái kết thúc. Logic trong mô-đun này được thiết kế để phương thức này chỉ được gọi trên các trạng thái kết thúc.

* `terminal_test(self, state)`: Trả về `True` nếu trạng thái hiện tại là trạng thái kết thúc (trò chơi dừng lại), và `False` nếu ngược lại.

* `to_move(self, state)`: Trả về người chơi có lượt đi tiếp theo từ trạng thái này. Thông tin này thường được lưu trữ sẵn trong trạng thái trò chơi, vì vậy phương thức này chỉ cần trích xuất và trả về nó.

* `display(self, state)`: In hoặc hiển thị trạng thái hiện tại của trò chơi ra màn hình.

# CÁC VÍ DỤ VỀ TRÒ CHƠI

Dưới đây là một số ví dụ về các trò chơi mà bạn có thể tạo và thử nghiệm.

## Tic-Tac-Toe

Hãy xem qua lớp `TicTacToe`. Tất cả các phương thức được đề cập trong lớp `Game` đều đã được triển khai đầy đủ tại đây.

In [35]:
%psource TicTacToe

Lớp `TicTacToe` được kế thừa từ lớp `Game`. Như đã đề cập trước đó, việc làm này là vô cùng cần thiết giúp việc phát hiện lỗi trở nên dễ dàng hơn nhiều.

Các phương thức bổ sung trong lớp `TicTacToe`:

* `__init__(self, h=3, v=3, k=3)`: Khi bạn tạo một lớp kế thừa từ lớp `Game` (trong trường hợp của chúng ta là lớp `TicTacToe`), bạn sẽ phải khởi tạo một đối tượng của lớp này để bắt đầu trò chơi. Quá trình khởi tạo này có thể yêu cầu một số thông tin bổ sung được truyền qua các biến của hàm `__init__`. Đối với trò chơi `TicTacToe`, thông tin bổ sung này sẽ bao gồm số hàng `h`, số cột `v` và số lượng ký tự X hoặc O liên tiếp cần thiết trên một hàng, cột hoặc đường chéo để giành chiến thắng `k`. Ngoài ra, trạng thái ban đầu của trò chơi cũng phải được định nghĩa tại đây.

* `compute_utility(self, board, move, player)`: Phương thức tính toán giá trị hữu dụng của trò chơi Tic-Tac-Toe. Nếu người chơi 'X' thắng sau nước đi này, phương thức trả về 1; nếu 'O' thắng trả về -1; ngược lại trả về 0.

* `k_in_row(self, board, move, player, delta_x_y)`: Phương thức này trả về `True` nếu có một đường thẳng (chiến thắng) được tạo ra trên bàn cờ với nước đi mới nhất, ngược lại trả về `False`.

### Trạng thái Trò chơi TicTacToe

Trước khi bắt đầu triển khai trò chơi `TicTacToe`, chúng ta cần quyết định cách biểu diễn trạng thái của trò chơi. Thông thường, một trạng thái trò chơi sẽ cung cấp cho bạn toàn bộ thông tin hiện tại tại bất kỳ thời điểm nào. Khi nhận được một trạng thái trò chơi, bạn phải biết lượt tiếp theo thuộc về ai, bàn cờ thực tế trông như thế nào, v.v. Một trạng thái trò chơi không nhất thiết phải lưu trữ lịch sử các nước đi trước đó. Nếu bạn vẫn có thể tiếp tục chơi dựa trên trạng thái hiện tại, thì cách biểu diễn trạng thái đó là chấp nhận được. Mặc dù chúng ta muốn lưu trữ đầy đủ thông tin, nhưng không nên đưa quá nhiều thông tin thừa vào trạng thái vì việc cập nhật nó để tạo ra trạng thái mới sẽ rất phức tạp.

Đối với trò chơi `TicTacToe`, liệu việc chỉ lưu trữ vị trí của tất cả các chữ X và O có đủ để đại diện cho toàn bộ thông tin trò chơi tại thời điểm đó không? Việc này có cho biết lượt tiếp theo là của ai không? Nhìn vào số lượng chữ X và O trên bàn cờ và đếm chúng có thể giúp ta biết được điều đó, nhưng cách này sẽ tốn thêm chi phí tính toán. Để tránh việc này, chúng ta sẽ lưu trữ trực tiếp thông tin người chơi tiếp theo `to_move` vào trong namedtuple `GameState`.

Từ việc này, chúng ta đã giảm bớt khối lượng tính toán bằng cách lưu trữ thêm một chút thông tin phụ trong trạng thái trò chơi. Chúng ta sẽ áp dụng kỹ thuật này nhiều hơn ở các phần sau.

Để lưu trữ trạng thái trò chơi, chúng ta sẽ sử dụng namedtuple `GameState`.

* `to_move`: Một chuỗi ký tự đơn, có giá trị là 'X' hoặc 'O'.

* `utility`: Trả về 1 nếu thắng, -1 nếu thua, và 0 nếu hòa.

* `board`: Lưu trữ tất cả các vị trí của X và O trên bàn cờ hiện tại.

* `moves`: Tất cả các nước đi khả thi từ trạng thái hiện tại. Lưu ý rằng việc lưu trữ các nước đi dưới dạng một danh sách như ở đây làm tăng độ phức tạp không gian của Tìm kiếm Minimax từ `O(m)` lên `O(bm)`. Hãy tham khảo Mục 5.2.1 trong sách.

### Biểu diễn một nước đi trong trò chơi TicTacToe

Bây giờ chúng ta đã quyết định cách biểu diễn trạng thái trò chơi, tiếp theo là xác định cách biểu diễn một nước đi để dễ dàng cập nhật trạng thái cũ sang trạng thái mới.

Đối với trò chơi `TicTacToe`, chúng ta biểu diễn một nước đi đơn giản bằng một tuple `(x, y)` tương ứng với chỉ số hàng và cột mà nước đi tiếp theo sẽ được thực hiện. Việc đi quân 'X' hay 'O' sẽ tự động được quyết định bởi giá trị `to_move` có sẵn trong `GameState`.

## Trò chơi trong Hình 5.2

Để làm quen với một ví dụ đơn giản hơn, chúng ta sẽ biểu diễn trò chơi được mô tả trong **Hình 5.2** của cuốn sách.

<img src="https://github.com/tranthanhthangbmt/webTTNT_2026A/blob/main/codeAndExercises/aima-python-master/notebooks/images/fig_5_2.png?raw=1" width="75%">

Các trạng thái được biểu diễn bằng các chữ cái viết hoa bên trong hình tam giác (ví dụ: "A"), trong khi các nước đi là các nhãn nằm trên các cạnh nối giữa các trạng thái (ví dụ: "a1"). Các nút lá chứa các giá trị hữu dụng (utility). Lưu ý rằng các nút lá trong ví dụ này được đặt tên là 'B1', 'B2', 'B3' cho các nút bên dưới 'B', và tương tự cho các nút khác.

Chúng ta sẽ mô hình hóa các nước đi, giá trị hữu dụng và trạng thái ban đầu như sau:

In [36]:
moves = dict(A=dict(a1='B', a2='C', a3='D'),
                 B=dict(b1='B1', b2='B2', b3='B3'),
                 C=dict(c1='C1', c2='C2', c3='C3'),
                 D=dict(d1='D1', d2='D2', d3='D3'))
utils = dict(B1=3, B2=12, B3=8, C1=2, C2=4, C3=6, D1=14, D2=5, D3=2)
initial = 'A'

Trong biến `moves`, chúng ta sử dụng một hệ thống dictionary lồng nhau. Dictionary bên ngoài có khóa (key) là các trạng thái hiện tại và giá trị (value) là các nước đi khả thi từ trạng thái đó (dưới dạng một dictionary khác). Dictionary bên trong có khóa là tên nước đi và giá trị là trạng thái tiếp theo đạt được sau khi thực hiện nước đi đó.

Dưới đây là một ví dụ minh họa cách sử dụng `moves`. Chúng ta muốn biết trạng thái tiếp theo sau khi thực hiện nước đi 'a1' từ trạng thái 'A', kết quả thu được là 'B'. Quan sát nhanh hình ảnh phía trên xác nhận điều này là hoàn toàn chính xác.

In [37]:
print(moves['A']['a1'])

B


Bây giờ chúng ta sẽ xem xét các hàm cần triển khai. Đầu tiên, chúng ta cần khởi tạo một đối tượng của lớp `Fig52Game`.

In [38]:
fig52 = Fig52Game()

`actions`: Trả về danh sách các nước đi có thể thực hiện từ một trạng thái cho trước.

In [39]:
psource(Fig52Game.actions)

In [40]:
print(fig52.actions('B'))

['b1', 'b2', 'b3']


`result`: Trả về trạng thái tiếp theo sau khi thực hiện một nước đi cụ thể.

In [41]:
psource(Fig52Game.result)

In [42]:
print(fig52.result('A', 'a1'))

B


`utility`: Trả về giá trị của trạng thái kết thúc đối với một người chơi ('MAX' hoặc 'MIN'). Lưu ý rằng đối với người chơi 'MIN', giá trị trả về sẽ là giá trị âm của utility.

In [43]:
psource(Fig52Game.utility)

In [44]:
print(fig52.utility('B1', 'MAX'))
print(fig52.utility('B1', 'MIN'))

3
-3


`terminal_test`: Trả về `True` nếu trạng thái hiện tại là trạng thái kết thúc, ngược lại trả về `False`.

In [45]:
psource(Fig52Game.terminal_test)

In [46]:
print(fig52.terminal_test('C3'))

True


`to_move`: Trả về người chơi sẽ thực hiện nước đi tại trạng thái này.

In [47]:
psource(Fig52Game.to_move)

In [48]:
print(fig52.to_move('A'))

MAX


Dưới đây là toàn bộ mã nguồn của lớp `Fig52Game` được kế thừa từ lớp `Game` và ghi đè các hàm của nó:

In [49]:
psource(Fig52Game)

# THUẬT TOÁN MIN-MAX

## Tổng quan

Thuật toán này (thường gọi là *Minimax*) tính toán nước đi tiếp theo tối ưu cho một người chơi (MIN hoặc MAX) tại trạng thái hiện tại của họ. Nó duyệt đệ quy để tính toán giá trị minimax của các trạng thái kế tiếp cho đến khi chạm tới các trạng thái kết thúc (các nút lá của cây quyết định). Sử dụng giá trị `utility` của các trạng thái kết thúc này, nó tính ngược lại giá trị cho các nút cha cho đến khi quay về nút gốc ban đầu.

Cần lưu ý rằng thuật toán này hoạt động theo cơ chế tìm kiếm theo chiều sâu (DFS). Bạn có thể xem mã giả của thuật toán bên dưới:

In [50]:
pseudocode("Minimax-Decision")

### AIMA3e
__function__ MINIMAX-DECISION(_state_) __returns__ _an action_  
&emsp;__return__ arg max<sub> _a_ &Element; ACTIONS(_s_)</sub> MIN\-VALUE(RESULT(_state_, _a_))  

---
__function__ MAX\-VALUE(_state_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &minus;&infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MAX(_v_, MIN\-VALUE(RESULT(_state_, _a_)))  
&emsp;__return__ _v_  

---
__function__ MIN\-VALUE(_state_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MIN(_v_, MAX\-VALUE(RESULT(_state_, _a_)))  
&emsp;__return__ _v_  

---
__Figure__ ?? An algorithm for calculating minimax decisions. It returns the action corresponding to the best possible move, that is, the move that leads to the outcome with the best utility, under the assumption that the opponent plays to minimize utility. The functions MAX\-VALUE and MIN\-VALUE go through the whole game tree, all the way to the leaves, to determine the backed\-up value of a state. The notation argmax <sub>_a_ &Element; _S_</sub> _f_(_a_) computes the element _a_ of set _S_ that has maximum value of _f_(_a_).

---
__function__ EXPECTIMINIMAX(_s_) =     
&emsp;UTILITY(_s_) __if__ TERMINAL\-TEST(_s_)  
&emsp;max<sub>_a_</sub> EXPECTIMINIMAX(RESULT(_s, a_)) __if__ PLAYER(_s_)= MAX  
&emsp;min<sub>_a_</sub> EXPECTIMINIMAX(RESULT(_s, a_)) __if__ PLAYER(_s_)= MIN  
&emsp;∑<sub>_r_</sub> P(_r_) EXPECTIMINIMAX(RESULT(_s, r_)) __if__ PLAYER(_s_)= CHANCE

## Triển khai

Trong phần triển khai, chúng ta sử dụng hai hàm chính là `max_value` và `min_value` để lần lượt tính toán nước đi tốt nhất cho người chơi MAX và MIN. Hai hàm này gọi lẫn nhau thông qua đệ quy luân phiên cho đến khi đạt tới trạng thái kết thúc. Khi quá trình đệ quy dừng lại, chúng ta sẽ có điểm số cho mỗi nước đi và chọn ra nước đi có điểm số lớn nhất. Mặc dù trả về giá trị max, thuật toán vẫn hoạt động đúng cho người chơi MIN vì đối với MIN, các giá trị hữu dụng đã được đảo dấu âm (do đó giá trị càng lớn thì càng có lợi cho MIN).

In [51]:
psource(minmax_decision)

## Ví dụ

Bây giờ chúng ta sẽ chạy thử trò chơi Fig52 bằng thuật toán này. Hãy quan sát sơ đồ Fig52Game ở trên để dễ hình dung.

Hiện tại đang là lượt đi của MAX và người chơi đang ở trạng thái A. MAX có thể chọn đi đến B, C hoặc D bằng các nước đi tương ứng là a1, a2 và a3. Mục tiêu của MAX là tối đa hóa giá trị nhận được cuối cùng. Do đó, để đưa ra quyết định, MAX cần biết giá trị của các nút kế tiếp và chọn giá trị lớn nhất. Sau lượt của MAX sẽ là lượt của MIN. Vì vậy, MAX cần biết các giá trị của B, C và D sẽ là bao nhiêu sau khi MIN thực hiện lượt chơi của mình.

Bài toán lúc này chuyển thành: MIN sẽ chọn nước đi nào tại các trạng thái B, C và D. Các trạng thái kế tiếp của các nút này đều là trạng thái kết thúc (nút lá), do đó MIN sẽ chọn giá trị nhỏ nhất cho mỗi nút. Cụ thể, đối với B, MIN chọn 3 (từ nước đi b1); đối với C, MIN chọn 2 (từ nước đi c1); và đối với D, MIN tiếp tục chọn 2 (từ nước đi d3).

Hãy cùng xem kết quả chạy thực tế bằng code:

In [52]:
print(minmax_decision('B', fig52))
print(minmax_decision('C', fig52))
print(minmax_decision('D', fig52))

b1
c1
d3


Bây giờ MAX đã biết giá trị nhận được tại các nhánh B, C và D lần lượt là 3, 2 và 2 (do các nước đi tối ưu của MIN tạo ra). Giá trị lớn nhất là 3, tương ứng với nước đi a1. Do đó, MAX sẽ quyết định chọn nước đi a1. Hãy cùng xem thuật toán hoạt động đầy đủ:

In [53]:
print(minmax_decision('A', fig52))

a1


## Trực quan hóa

Dưới đây là một công cụ trực quan hóa trò chơi đơn giản sử dụng thuật toán Minimax. Sau khi chạy dòng lệnh, hãy nhấp chuột vào ô hiển thị để bắt đầu trò chơi. Bạn cũng có thể tự nhập danh sách các giá trị của riêng mình thông qua một mảng gồm 27 số nguyên.

In [54]:
from aima.notebook_utils import Canvas_min_max
from random import randint

In [55]:
minimax_viz = Canvas_min_max('minimax_viz', [randint(1, 50) for i in range(27)])

# CẮT TỈA ALPHA-BETA

## Tổng quan

Mặc dù thuật toán *Minimax* rất hiệu quả để tìm nước đi tối ưu, tuy nhiên nó sẽ gặp khó khăn lớn khi không gian trạng thái của trò chơi mở rộng. Thuật toán bắt buộc phải duyệt qua tất cả các nút lá của cây, vốn tăng theo cấp số nhân dựa trên độ sâu của cây.

Đối với Tic-Tac-Toe, độ sâu tối đa của cây là 9 (sau nước đi thứ 9 trò chơi chắc chắn kết thúc), chúng ta có tối đa 9! trạng thái kết thúc (thực tế ít hơn vì trò chơi có thể kết thúc sớm hơn). Con số này không quá lớn, nhưng với các trò chơi phức tạp hơn như cờ vua, chúng ta có tới hơn $10^{40}$ nút lá. Hiện tại chưa có cách nào để loại bỏ hoàn toàn số mũ này, nhưng chúng ta có các phương pháp giúp giảm bớt đáng kể khối lượng công việc tìm kiếm.

Ở đây chúng ta sẽ tìm hiểu phương pháp *cắt tỉa* cây trò chơi, nghĩa là loại bỏ các nhánh của cây mà chúng ta chắc chắn không cần phải duyệt qua. Kỹ thuật cắt tỉa cụ thể này được gọi là *alpha-beta*, và quá trình tìm kiếm kết hợp nó gọi là *tìm kiếm alpha-beta*.

Để minh họa những phần nào của cây không cần tìm kiếm, chúng ta hãy xem lại ví dụ `Fig52Game`.

Trong trò chơi ví dụ này, chúng ta cần tìm nước đi tốt nhất cho người chơi MAX tại trạng thái A, đó là giá trị lớn nhất trong số các nước đi tiếp theo của MIN tại các trạng thái kế cận.

`MAX(A) = MAX( MIN(B), MIN(C), MIN(D) )`

Chúng ta đã biết `MIN(B)` là giá trị nhỏ nhất trong số 3, 12, 8, tức là bằng 3. Công thức trên trở thành:

`MAX(A) = MAX( 3, MIN(C), MIN(D) )`

Nước đi tiếp theo chúng ta kiểm tra là c1, dẫn đến trạng thái kết thúc có giá trị hữu dụng là 2. Trước khi tiếp tục tìm kiếm sâu hơn dưới nhánh C, hãy thay giá trị mới này vào công thức:

`MAX(A) = MAX( 3, MIN(2, c2, .... cN), MIN(D) )`

Chúng ta chưa biết trạng thái C cho phép bao nhiêu nước đi tiếp theo, nhưng chúng ta biết chắc chắn rằng nước đi đầu tiên của nhánh này có giá trị là 2. Liệu chúng ta có cần tiếp tục tìm kiếm các nhánh khác dưới C không? Câu trả lời là KHÔNG. Giá trị mà MIN sẽ chọn tại C chắc chắn sẽ nhỏ hơn hoặc bằng 2. Vì MAX đã có một lựa chọn tốt hơn ở nhánh khác với giá trị bằng 3 (từ nhánh B), nên MAX hoàn toàn không cần tốn thời gian tìm kiếm thêm các nhánh con khác dưới C nữa.

Trong thuật toán *alpha-beta*, chúng ta sử dụng thêm hai tham số cho mỗi trạng thái/nút là *alpha (a)* và *beta (b)* để mô tả các giới hạn nước đi. Tham số *a* đại diện cho lựa chọn tốt nhất hiện tại (giá trị cao nhất) đối với người chơi MAX dọc theo đường đi, trong khi *b* đại diện cho lựa chọn tốt nhất (giá trị thấp nhất) đối với người chơi MIN. Khi duyệt cây, chúng ta liên tục cập nhật *a* và *b*, đồng thời tiến hành cắt tỉa (ngừng tìm kiếm) một nhánh khi phát hiện giá trị tại nút đó tệ hơn giới hạn *a* hoặc *b* hiện tại.

Trong ví dụ trên, sau khi tìm kiếm xong nhánh B, người chơi MAX đã có giá trị *a* tối thiểu là 3. Do đó, khi tìm kiếm dưới nút C và phát hiện ra giá trị 2 (nhỏ hơn 3), chúng ta dừng ngay việc tìm kiếm các nhánh còn lại của C.

Bạn có thể đọc mã giả của thuật toán bên dưới:

In [56]:
pseudocode("Alpha-Beta-Search")

### AIMA3e
__function__ ALPHA-BETA-SEARCH(_state_) __returns__ an action  
&emsp;_v_ &larr; MAX\-VALUE(_state_, &minus;&infin;, &plus;&infin;)  
&emsp;__return__ the _action_ in ACTIONS(_state_) with value _v_  

---
__function__ MAX\-VALUE(_state_, _&alpha;_, _&beta;_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &minus;&infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MAX(_v_, MIN\-VALUE(RESULT(_state_, _a_), _&alpha;_, _&beta;_))  
&emsp;&emsp;&emsp;__if__ _v_ &ge; _&beta;_ __then return__ _v_  
&emsp;&emsp;&emsp;_&alpha;_ &larr; MAX(_&alpha;_, _v_)  
&emsp;__return__ _v_  

---
__function__ MIN\-VALUE(_state_, _&alpha;_, _&beta;_) __returns__ _a utility value_  
&emsp;__if__ TERMINAL\-TEST(_state_) __then return__ UTILITY(_state_)  
&emsp;_v_ &larr; &plus;&infin;  
&emsp;__for each__ _a_ __in__ ACTIONS(_state_) __do__  
&emsp;&emsp;&emsp;_v_ &larr; MIN(_v_, MAX\-VALUE(RESULT(_state_, _a_), _&alpha;_, _&beta;_))  
&emsp;&emsp;&emsp;__if__ _v_ &le; _&alpha;_ __then return__ _v_  
&emsp;&emsp;&emsp;_&beta;_ &larr; MIN(_&beta;_, _v_)  
&emsp;__return__ _v_  


---
__Figure__ ?? The alpha\-beta search algorithm. Notice that these routines are the same as the MINIMAX functions in Figure ??, except for the two lines in each of MIN\-VALUE and MAX\-VALUE that maintain _&alpha;_ and _&beta;_ (and the bookkeeping to pass these parameters along).

## Triển khai

Tương tự như *minimax*, chúng ta tiếp tục sử dụng hai hàm `max_value` và `min_value`, nhưng lần này sẽ kết hợp thêm các giá trị *a* và *b*, cập nhật chúng liên tục và dừng các cuộc gọi đệ quy nếu gặp phải các nút có giá trị tệ hơn giới hạn cho phép. Thuật toán cuối cùng sẽ trả về nước đi mang lại giá trị tối ưu nhất.

Chi tiết triển khai code:

In [57]:
%psource alpha_beta_search

## Ví dụ

Chúng ta sẽ chơi thử trò chơi Fig52 bằng thuật toán tìm kiếm *alpha-beta*. Hiện tại là lượt đi của MAX xuất phát từ trạng thái A.

In [58]:
print(alpha_beta_search('A', fig52))

a1


Nước đi tối ưu nhất cho MAX là a1 vì các lý do đã phân tích ở trên. MIN sẽ chọn nước đi b1 cho nhánh B thu được giá trị là 3, cập nhật giá trị giới hạn *a* của MAX lên 3. Sau đó, khi tìm kiếm dưới nhánh C và gặp nút có giá trị 2, thuật toán sẽ bỏ qua toàn bộ phần còn lại của nhánh C vì nó nhỏ hơn *a*. Ở nhánh D chúng ta cũng thu được giá trị 2. Kết quả cuối cùng, nước đi tốt nhất cho MAX là nước đi dẫn đến giá trị bằng 3, tức là nước đi a1.

Dưới đây là các nước đi tốt nhất của MIN khi bắt đầu từ các vị trí B, C và D. Lưu ý rằng trong các trường hợp này thuật toán hoạt động tương tự như *minimax* vì tất cả các nút con bên dưới chúng đều là trạng thái kết thúc.

In [59]:
print(alpha_beta_search('B', fig52))
print(alpha_beta_search('C', fig52))
print(alpha_beta_search('D', fig52))

b1
c1
d3


## Trực quan hóa

Dưới đây là phần trực quan hóa thuật toán alpha-beta đối với một trò chơi đơn giản. Hãy nhấp chuột vào ô hiển thị sau khi chạy lệnh để theo dõi từng bước của trò chơi. Bạn có thể tự định nghĩa các giá trị của các nút lá thông qua một mảng gồm 27 số nguyên.

In [60]:
from aima.notebook_utils import Canvas_alpha_beta
from random import randint

In [61]:
alphabeta_viz = Canvas_alpha_beta('alphabeta_viz', [randint(1, 50) for i in range(27)])

# NGƯỜI CHƠI (PLAYERS)

Chúng ta đã hoàn thành việc triển khai các lớp `TicTacToe` và `Fig52Game` để định nghĩa luật chơi. Tuy nhiên, để tạo ra một thực thể AI có thể thực sự tham gia chơi game, chúng ta cần sử dụng các định nghĩa người chơi như `random_player` hay `alphabeta_player`.

## query_player
Phương thức `query_player` cho phép bạn (một người chơi thực tế) thực hiện nước đi tương tác với trò chơi. Phương thức này yêu cầu lớp trò chơi phải cài đặt sẵn hàm `display` để trạng thái bàn cờ có thể liên tục hiển thị trên màn hình terminal giúp bạn dễ dàng quan sát và tính toán nước đi.

## random_player
`random_player` là người chơi thực hiện các nước đi một cách hoàn toàn ngẫu nhiên. Không có bất kỳ chiến thuật phức tạp nào ở người chơi này.

## alphabeta_player
Ngược lại, `alphabeta_player` sẽ gọi hàm tìm kiếm `alphabeta_search` để luôn trả về nước đi tối ưu nhất từ trạng thái hiện tại. Người chơi này luôn đảm bảo đi nước cờ hoàn hảo nhất với điều kiện cây trò chơi đủ nhỏ để có thể duyệt hết.

## minimax_player
Người chơi `minimax_player` sẽ gọi thuật toán `minimax_search` để đưa ra nước đi tốt nhất từ trạng thái hiện tại.

## play_game
Hàm `play_game` là hàm cốt lõi dùng để vận hành trò chơi. Bạn chỉ cần truyền vào đối tượng trò chơi muốn chơi cùng cấu hình các người chơi tham gia. Bạn có thể thiết lập các trận đấu giữa AI vs AI, AI vs Người, hoặc thậm chí là Người vs Người!

# HÃY CÙNG CHƠI THỬ MỘT SỐ TRÒ CHƠI!

## Game52

Hãy bắt đầu thử nghiệm với trò chơi `Fig52Game` trước. Chúng ta sẽ tạo ra một đối tượng đại diện cho trò chơi này từ lớp con được kế thừa từ `Game`:

In [62]:
game52 = Fig52Game()

Đầu tiên, hãy thử nghiệm với `random_player(game, state)`. Với mỗi trạng thái trò chơi được truyền vào, người chơi này sẽ đưa ra một nước đi ngẫu nhiên khác nhau:

In [63]:
print(random_player(game52, 'A'))
print(random_player(game52, 'A'))

a2
a1


`alphabeta_player(game, state)` sẽ luôn trả về nước đi tốt nhất cho người chơi tương ứng (MAX hoặc MIN):

In [64]:
print( alpha_beta_player(game52, 'A') )
print( alpha_beta_player(game52, 'B') )
print( alpha_beta_player(game52, 'C') )

a1
b1
c1


Thực chất, những gì `alphabeta_player` làm chỉ là gọi hàm `alphabeta_full_search`. Về bản chất cả hai đều giống nhau. Trong thư viện này, cả `alphabeta_full_search` và `minimax_decision` đều đã được cài đặt. Chúng đều thực hiện chung một nhiệm vụ và trả về kết quả giống nhau (nước đi tối ưu nhất). Điểm khác biệt duy nhất là `alphabeta_full_search` hoạt động hiệu quả hơn rất nhiều về mặt thời gian nhờ lược bỏ bớt các trạng thái không cần thiết.

In [65]:
minmax_decision('A', game52)

'a1'

In [66]:
alpha_beta_search('A', game52)

'a1'

Chạy thử nghiệm hàm `play_game` trên trò chơi game52:

In [67]:
game52.play_game(alpha_beta_player, alpha_beta_player)

B1


3

In [68]:
game52.play_game(alpha_beta_player, random_player)

B1


3

In [69]:
game52.play_game(query_player, alpha_beta_player)

current state:
A
available moves: ['a1', 'a2', 'a3']

Your move? a1
B1


3

In [70]:
game52.play_game(alpha_beta_player, query_player)

current state:
B
available moves: ['b1', 'b2', 'b3']

Your move? b1
B1


3

Lưu ý rằng nếu bạn là người đi trước thì `alphabeta_player` đóng vai trò là MIN, còn nếu bạn đi sau thì `alphabeta_player` đóng vai trò là MAX. Điều này được quy định trực tiếp trong cách thiết lập luật chơi của lớp `Fig52Game`. Bạn có thể xem lại mã nguồn của lớp này để hiểu rõ hơn.

## TicTacToe

Bây giờ chúng ta sẽ chuyển sang trò chơi `TicTacToe`. Đầu tiên, khởi tạo trò chơi bằng cách tạo một thực thể của lớp con `TicTacToe` kế thừa từ `Game`:

In [71]:
ttt = TicTacToe()

Chúng ta có thể in trạng thái bàn cờ hiện tại ra màn hình bằng phương thức `display`:

In [72]:
ttt.display(ttt.initial)

. . . 
. . . 
. . . 


Như vậy đây chính là trạng thái khởi đầu của trò chơi: một bàn cờ trống hoàn toàn chưa có quân X hay O nào.

Hãy cùng tự tạo ra một trạng thái bàn cờ mới để làm các thí nghiệm tiếp theo:

In [73]:
my_state = GameState(
    to_move = 'X',
    utility = '0',
    board = {(1,1): 'X', (1,2): 'O', (1,3): 'X',
             (2,1): 'O',             (2,3): 'O',
             (3,1): 'X',
            },
    moves = [(2,2), (3,2), (3,3)]
    )

Trạng thái bàn cờ này trông như thế nào?

In [74]:
ttt.display(my_state)

X O X 
O . . 
X O . 


Người chơi ngẫu nhiên `random_player` sẽ đưa ra nước đi hoàn toàn ngẫu nhiên trong số các ô còn trống:

In [75]:
random_player(ttt, my_state)

(2, 2)

In [76]:
random_player(ttt, my_state)

(3, 3)

Trong khi đó, `alphabeta_player` sẽ tính toán và luôn luôn đưa ra nước đi tối ưu nhất:

In [77]:
alpha_beta_player(ttt, my_state)

(2, 2)

Bây giờ hãy thử cho hai người chơi thi đấu trực tiếp với nhau bằng hàm `play_game`. Hàm này sẽ điều phối trận đấu và trả về kết quả hữu dụng (utility) của người chơi thứ nhất khi trận đấu kết thúc. Đối với Tic-Tac-Toe, nếu kết quả trả về là +1 nghĩa là người đi trước thắng, -1 là người đi sau thắng, và 0 tương ứng với kết quả hòa.

In [78]:
ttt.play_game(random_player, alpha_beta_player)

. . X 
O O O 
X . X 


-1

Kết quả thông thường sẽ là -1 do `random_player` rất dễ bị đánh bại bởi `alphabeta_player`. Tuy nhiên, thỉnh thoảng người chơi ngẫu nhiên vẫn có thể may mắn thủ hòa được.

Vì `alphabeta_player` chơi một cách hoàn hảo, trận đấu giữa hai người chơi `alphabeta_player` chắc chắn sẽ luôn kết thúc với kết quả hòa. Chúng ta cùng kiểm chứng điều này qua 10 ván đấu thử nghiệm:

In [79]:
for _ in range(10):
    print(ttt.play_game(alpha_beta_player, alpha_beta_player))

X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0
X O X 
X O O 
O X X 
0


Ngược lại, một người chơi ngẫu nhiên `random_player` sẽ không bao giờ có cơ hội giành chiến thắng trước đối thủ hoàn hảo `alphabeta_player`. Hãy cùng kiểm tra thử:

In [80]:
for _ in range(10):
    print(ttt.play_game(random_player, alpha_beta_player))

O O X 
X O . 
X X O 
-1
O . X 
O . . 
O X X 
-1
X O X 
O X X 
O X O 
0
O . . 
O X X 
O X . 
-1
O O X 
X X O 
O X X 
0
O O . 
O X X 
O X X 
-1
O X O 
X O X 
O X . 
-1
O . X 
X O X 
. . O 
-1
O O O 
X X . 
O X X 
-1
O X X 
X O O 
O X X 
0


## Canvas_TicTacToe(Canvas)

Lớp con này được sử dụng để bạn có thể chơi game TicTacToe một cách trực quan và tương tác trực tiếp ngay trong Jupyter Notebook. Lớp TicTacToe sẽ được gọi tự động khi khởi tạo thành phần này.

Hãy thử nghiệm một trận đấu tự động giữa `random_player` và `alphabeta_player`. Bạn có thể nhấp chuột lên bàn cờ để yêu cầu các người chơi máy thực hiện nước đi tiếp theo.

In [81]:
from aima.notebook_utils import Canvas_TicTacToe

In [82]:
bot_play = Canvas_TicTacToe('bot_play', 'random', 'alpha_beta')

Bây giờ, hãy tự mình trải nghiệm trận đấu với đối thủ máy chơi ngẫu nhiên `random_player`:

In [83]:
rand_play = Canvas_TicTacToe('rand_play', 'human', 'random')

Chúc mừng! Thông thường chúng ta sẽ dễ dàng giành chiến thắng. Tuy nhiên, bạn sẽ thấy rằng mình không bao giờ có thể thắng được đối thủ thông minh `alphabeta_player` dù có cố gắng đến đâu.

In [84]:
ab_play = Canvas_TicTacToe('ab_play', 'human', 'alpha_beta')